# Admission Mortality - Inpatient

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl
import numpy as np

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics, preprocess_inference_results, compute_ci
from ethos.utils import get_mimic_hf_patients, get_mimic_sepsis_icu_stays

# Requires original MIMIC-IV 2.2 dataset that we do not provide
mimic_sepsis = get_mimic_sepsis_icu_stays()
# heart_failure_cohort = get_mimic_hf_patients(PROJECT_ROOT / "data/mimic-2.2")

In [ ]:
results_dir = PROJECT_ROOT / "results"
hosp_mort_result_dir = results_dir / Task.HOSPITAL_MORTALITY

def process_hosp_mortality_results(input_dir, outcome_tokens) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual").is_in(outcome_tokens),
        expected_expr=pl.col("expected").is_in(outcome_tokens),
        filter_ambiguous=(
            pl.col("actual").is_in([ST.DEATH, ST.DISCHARGE])
            & (pl.col("stop_reason") == Reason.GOT_TOKEN)
        ),
    )

In [ ]:
for input_dir in (results_dir / Task.HOSPITAL_MORTALITY).iterdir():
    df = process_hosp_mortality_results(input_dir, outcome_tokens=[ST.DEATH])

    results = compute_and_print_metrics(
        df["expected"].to_numpy(),
        df["actual"].to_numpy(),
        figure_title="Hospital Mortality (ETHOS)",
    )

In [ ]:
metrics = ["accuracy", "sensitivity", "specificity", "precision", "tp", "tn", "fp", "fn"]
for metric in metrics:
    res = results[metric]
    if type(res) is np.float64:
        print(f"{metric}: {res:.3f}")
    else:
        print(f"{metric}: {res}")